In [183]:
import pandas as pd 
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [25]:
anime = pd.read_csv('anime-dataset-2023.csv')

In [26]:
anime.columns 

Index(['anime_id', 'Name', 'English name', 'Other name', 'Score', 'Genres',
       'Synopsis', 'Type', 'Episodes', 'Aired', 'Premiered', 'Status',
       'Producers', 'Licensors', 'Studios', 'Source', 'Duration', 'Rating',
       'Rank', 'Popularity', 'Favorites', 'Scored By', 'Members', 'Image URL'],
      dtype='object')

In [29]:
anime.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24905 entries, 0 to 24904
Data columns (total 24 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   anime_id      24905 non-null  int64 
 1   Name          24905 non-null  object
 2   English name  24905 non-null  object
 3   Other name    24905 non-null  object
 4   Score         24905 non-null  object
 5   Genres        24905 non-null  object
 6   Synopsis      24905 non-null  object
 7   Type          24905 non-null  object
 8   Episodes      24905 non-null  object
 9   Aired         24905 non-null  object
 10  Premiered     24905 non-null  object
 11  Status        24905 non-null  object
 12  Producers     24905 non-null  object
 13  Licensors     24905 non-null  object
 14  Studios       24905 non-null  object
 15  Source        24905 non-null  object
 16  Duration      24905 non-null  object
 17  Rating        24905 non-null  object
 18  Rank          24905 non-null  object
 19  Popu

In [ ]:
anime.head()

In [35]:
anime[['English name','Genres']].head(5)

,English name,Genres
0,Cowboy Bebop,"Action, Award Winning, Sci-Fi"
1,Cowboy Bebop: The Movie,"Action, Sci-Fi"
2,Trigun,"Action, Adventure, Sci-Fi"
3,Witch Hunter Robin,"Action, Drama, Mystery, Supernatural"
4,Beet the Vandel Buster,"Adventure, Fantasy, Supernatural"


In [ ]:
# columns to keep = id 
# name
# English name
# genres
# popularity
# type
# source
# rating
# image_url
# studios
# score

In [37]:
df = anime[['anime_id', 'Name', 'English name', 'Genres', 'Studios', 'Type', 'Source', 'Rating', 'Score', 'Popularity', 'Image URL']].copy()

In [41]:
df.head(5)

,anime_id,Name,English name,Genres,Studios,Type,Source,Rating,Score,Popularity,Image URL
0,1,Cowboy Bebop,Cowboy Bebop,"Action, Award Winning, Sci-Fi",Sunrise,TV,Original,R - 17+ (violence & profanity),8.75,43,https://cdn.myanimelist.net/images/anime/4/196...
1,5,Cowboy Bebop: Tengoku no Tobira,Cowboy Bebop: The Movie,"Action, Sci-Fi",Bones,Movie,Original,R - 17+ (violence & profanity),8.38,602,https://cdn.myanimelist.net/images/anime/1439/...
2,6,Trigun,Trigun,"Action, Adventure, Sci-Fi",Madhouse,TV,Manga,PG-13 - Teens 13 or older,8.22,246,https://cdn.myanimelist.net/images/anime/7/203...
3,7,Witch Hunter Robin,Witch Hunter Robin,"Action, Drama, Mystery, Supernatural",Sunrise,TV,Original,PG-13 - Teens 13 or older,7.25,1795,https://cdn.myanimelist.net/images/anime/10/19...
4,8,Bouken Ou Beet,Beet the Vandel Buster,"Adventure, Fantasy, Supernatural",Toei Animation,TV,Manga,PG - Children,6.94,5126,https://cdn.myanimelist.net/images/anime/7/215...


In [57]:
df['Genres']=df['Genres'].str.replace(', ',' ')
df['Studios']=df['Studios'].fillna('').str.replace(', ',' ')
df['Type']=df['Type'].fillna('')
df['Source']=df['Source'].fillna('')

In [63]:
df.columns

Index(['anime_id', 'Name', 'English name', 'Genres', 'Studios', 'Type',
       'Source', 'Rating', 'Score', 'Popularity', 'Image URL'],
      dtype='object')

In [65]:
df.head(5)

,anime_id,Name,English name,Genres,Studios,Type,Source,Rating,Score,Popularity,Image URL
0,1,Cowboy Bebop,Cowboy Bebop,Action Award Winning Sci-Fi,Sunrise,TV,Original,R - 17+ (violence & profanity),8.75,43,https://cdn.myanimelist.net/images/anime/4/196...
1,5,Cowboy Bebop: Tengoku no Tobira,Cowboy Bebop: The Movie,Action Sci-Fi,Bones,Movie,Original,R - 17+ (violence & profanity),8.38,602,https://cdn.myanimelist.net/images/anime/1439/...
2,6,Trigun,Trigun,Action Adventure Sci-Fi,Madhouse,TV,Manga,PG-13 - Teens 13 or older,8.22,246,https://cdn.myanimelist.net/images/anime/7/203...
3,7,Witch Hunter Robin,Witch Hunter Robin,Action Drama Mystery Supernatural,Sunrise,TV,Original,PG-13 - Teens 13 or older,7.25,1795,https://cdn.myanimelist.net/images/anime/10/19...
4,8,Bouken Ou Beet,Beet the Vandel Buster,Adventure Fantasy Supernatural,Toei Animation,TV,Manga,PG - Children,6.94,5126,https://cdn.myanimelist.net/images/anime/7/215...


In [49]:
df.isnull().sum()

anime_id        0
Name            0
English name    0
Genres          0
Studios         0
Type            0
Source          0
Rating          0
Score           0
Popularity      0
Image URL       0
dtype: int64

In [69]:
df['Rating'].unique()

array(['R - 17+ (violence & profanity)', 'PG-13 - Teens 13 or older',
       'PG - Children', 'R+ - Mild Nudity', 'G - All Ages', 'Rx - Hentai',
       'UNKNOWN'], dtype=object)

In [73]:
rating_map = {
    'G - All Ages': 'G',
    'PG - Children': 'PG',
    'PG-13 - Teens 13 or older': 'PG13',
    'R - 17+ (violence & profanity)': 'R',
    'R+ - Mild Nudity': 'Rplus',
    'Rx - Hentai': 'Rx'
}

df['Rating'] = df['Rating'].map(rating_map)

In [79]:
df

,anime_id,Name,English name,Genres,Studios,Type,Source,Rating,Score,Popularity,Image URL
0,1,Cowboy Bebop,Cowboy Bebop,Action Award Winning Sci-Fi,Sunrise,TV,Original,R,8.75,43,https://cdn.myanimelist.net/images/anime/4/196...
1,5,Cowboy Bebop: Tengoku no Tobira,Cowboy Bebop: The Movie,Action Sci-Fi,Bones,Movie,Original,R,8.38,602,https://cdn.myanimelist.net/images/anime/1439/...
2,6,Trigun,Trigun,Action Adventure Sci-Fi,Madhouse,TV,Manga,PG13,8.22,246,https://cdn.myanimelist.net/images/anime/7/203...
3,7,Witch Hunter Robin,Witch Hunter Robin,Action Drama Mystery Supernatural,Sunrise,TV,Original,PG13,7.25,1795,https://cdn.myanimelist.net/images/anime/10/19...
4,8,Bouken Ou Beet,Beet the Vandel Buster,Adventure Fantasy Supernatural,Toei Animation,TV,Manga,PG,6.94,5126,https://cdn.myanimelist.net/images/anime/7/215...
...,...,...,...,...,...,...,...,...,...,...,...
24900,55731,Wu Nao Monu,UNKNOWN,Comedy Fantasy Slice of Life,UNKNOWN,ONA,Web manga,PG13,UNKNOWN,24723,https://cdn.myanimelist.net/images/anime/1386/...
24901,55732,Bu Xing Si: Yuan Qi,Blader Soul,Action Adventure Fantasy,UNKNOWN,ONA,Web novel,PG13,UNKNOWN,0,https://cdn.myanimelist.net/images/anime/1383/...
24902,55733,Di Yi Xulie,The First Order,Action Adventure Fantasy Sci-Fi,UNKNOWN,ONA,Web novel,PG13,UNKNOWN,0,https://cdn.myanimelist.net/images/anime/1130/...
24903,55734,Bokura no Saishuu Sensou,UNKNOWN,UNKNOWN,UNKNOWN,Music,Original,PG13,UNKNOWN,0,https://cdn.myanimelist.net/images/anime/1931/...


In [89]:
df['tags'] = df['Genres'] + ' ' + df['Studios'] + ' ' + df['Type'] + ' ' + df['Source'] + ' ' + df['Rating']
df['tags'] = df['tags'].str.lower()

In [91]:
df['tags'].head(5)

0    action award winning sci-fi sunrise tv original r
1                 action sci-fi bones movie original r
2       action adventure sci-fi madhouse tv manga pg13
3    action drama mystery supernatural sunrise tv o...
4    adventure fantasy supernatural toei animation ...
Name: tags, dtype: object

In [95]:
df = df[['anime_id','Name','English name','Score','Popularity','tags','Image URL']]

In [97]:
df.head(5)

,anime_id,Name,English name,Score,Popularity,tags,Image URL
0,1,Cowboy Bebop,Cowboy Bebop,8.75,43,action award winning sci-fi sunrise tv original r,https://cdn.myanimelist.net/images/anime/4/196...
1,5,Cowboy Bebop: Tengoku no Tobira,Cowboy Bebop: The Movie,8.38,602,action sci-fi bones movie original r,https://cdn.myanimelist.net/images/anime/1439/...
2,6,Trigun,Trigun,8.22,246,action adventure sci-fi madhouse tv manga pg13,https://cdn.myanimelist.net/images/anime/7/203...
3,7,Witch Hunter Robin,Witch Hunter Robin,7.25,1795,action drama mystery supernatural sunrise tv o...,https://cdn.myanimelist.net/images/anime/10/19...
4,8,Bouken Ou Beet,Beet the Vandel Buster,6.94,5126,adventure fantasy supernatural toei animation ...,https://cdn.myanimelist.net/images/anime/7/215...


In [117]:
df = df.dropna(subset=['tags'])
print(df.shape)

(24236, 7)


In [145]:
df = df[df['English name'] != 'UNKNOWN'].reset_index(drop=True)

In [167]:
df = df[df['Score'] != 'UNKNOWN'].reset_index(drop=True)
print(df.shape)

(8021, 7)


In [169]:
cv = CountVectorizer(max_features = 5000)

In [171]:
vector = cv.fit_transform(df['tags']).toarray()

In [173]:
vector

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int64)

In [175]:
vector.shape

(8021, 860)

In [177]:
similarity = cosine_similarity(vector)
print(similarity.shape)

(8021, 8021)


In [179]:
def recommend(anime_name,n=10):
     if anime_name not in df['English name'].values:
        return "Anime not found in dataset"
    index = df[df['English name']== anime_name].index[0]
    index = df.index.get_loc(index)
    
    scores = list(enumerate(similarity[index]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = scores[1:n+1]  # skip index 0 (itself)
    
    
    indices = [i[0] for i in scores]
    return df.iloc[indices][['English name', 'Score', 'Image URL']]

    
    

In [181]:
recommend('Naruto')

,English name,Score,Image URL
193,Bleach,7.92,https://cdn.myanimelist.net/images/anime/3/404...
1100,Naruto Shippuden,8.26,https://cdn.myanimelist.net/images/anime/1565/...
5431,Boruto: Naruto Next Generations,6.06,https://cdn.myanimelist.net/images/anime/1091/...
4412,Yona of the Dawn,8.03,https://cdn.myanimelist.net/images/anime/9/642...
6183,Ultramarine Magmell,6.1,https://cdn.myanimelist.net/images/anime/1063/...
171,Flame of Recca,7.34,https://cdn.myanimelist.net/images/anime/1646/...
282,Yu Yu Hakusho: Ghost Files,8.46,https://cdn.myanimelist.net/images/anime/1228/...
7003,Bleach: Thousand-Year Blood War,9.07,https://cdn.myanimelist.net/images/anime/1908/...
7276,Orient,6.6,https://cdn.myanimelist.net/images/anime/1576/...
87,Mysterious Play,7.61,https://cdn.myanimelist.net/images/anime/2/201...


In [187]:
pickle.dump(similarity,open('similarity.pkl','wb'))
pickle.dump(df,open('anime.pkl', 'wb'))

In [1]:
import os
print(os.path.getsize('similarity.pkl') / (1024*1024), "MB")
print(os.path.getsize('anime.pkl') / (1024*1024), "MB")

490.84824657440186 MB
1.3950634002685547 MB
